# einops + pycograd

[einops](https://einops.rocks) gives you readable tensor surgery — `rearrange`, `reduce`,
`repeat`, `einsum` — driven by named-axis patterns. pycograd is a small reverse-mode
autograd on numpy. This notebook shows they compose: einops operations run **directly on
pycograd `Var`s**, stay on the tape, and therefore differentiate and vectorize.

einops dispatches through a backend registry keyed by tensor type. pycograd registers a
`Var` backend automatically **once both `pycograd` and `einops` are imported** (in either
order) — no setup call needed. (If you ever need it explicitly: `pg.register_einops_backend()`.)

The examples below are ported from the einops
[PyTorch examples page](https://einops.rocks/pytorch-examples.html), written in pycograd's
`|>` pipe DSL.

In [1]:
%load_ext pycograd

In [2]:
import numpy as np
import einops
import pycograd as pg

rng = np.random.default_rng(0)
# a small batch of 'images': (batch, channels, height, width)
N, C, H, W, K = 64, 3, 8, 8, 3
X = rng.standard_normal((N, C, H, W))
labels = rng.integers(0, K, N)
Y = np.eye(K)[labels]   # one-hot targets, K classes

## Building blocks: `rearrange` / `reduce` / `repeat` on a `Var`

These are the same calls you'd make on a numpy or torch tensor — here the operand is a
pycograd `Var`, so the result is a tape node, not a detached array.

In [3]:
v = pg.Var(rng.standard_normal((2, 3, 4, 4)))

print('flatten      b c h w -> b (c h w) :', einops.rearrange(v, 'b c h w -> b (c h w)').shape)
print('to-sequence  b c h w -> b (h w) c :', einops.rearrange(v, 'b c h w -> b (h w) c').shape)
print('avg-pool     b c h w -> b c (mean):', einops.reduce(v, 'b c h w -> b c', 'mean').shape)
print('repeat       b c h w -> b c h (w r):', einops.repeat(v, 'b c h w -> b c h (w r)', r=2).shape)

# the einops result carries the same values as the plain-numpy computation
got = np.asarray(einops.rearrange(v, 'b c h w -> b (c h w)').value)
ref = einops.rearrange(np.asarray(v.value), 'b c h w -> b (c h w)')
print('matches numpy                     :', np.allclose(got, ref))

flatten      b c h w -> b (c h w) : (2, 48)
to-sequence  b c h w -> b (h w) c : (2, 16, 3)
avg-pool     b c h w -> b c (mean): (2, 3)
repeat       b c h w -> b c h (w r): (2, 3, 4, 8)
matches numpy                     : True


## Gradients flow through einops — a tiny classifier

Example 1 / 19 on the page flattens a conv feature map for a dense layer with
`rearrange('b c h w -> b (c h w)')`. We drop that straight into a `|>` forward pipeline and
train it: because the flatten is on the tape, `weights.grad` differentiates through it.

In [4]:
with params{
    w1 = 0.05 * rng.standard_normal((C * H * W, 32))
    b1 = np.zeros(32)
    w2 = 0.05 * rng.standard_normal((32, K))
    b2 = np.zeros(K)
} as weights:
    forward = $ |> einops.rearrange($, 'n c h w -> n (c h w)') |> $ @ w1 + b1 |> pg.relu |> $ @ w2 + b2
    objective = |> X |> forward |> pg.cross_entropy($, Y)
    for _ in range(150):
        value, grads = weights.grad(objective)
        weights.step(grads, 0.2)
    logits = forward(X)

acc = (np.argmax(np.asarray(getattr(logits, 'value', logits)), axis=1) == labels).mean()
print(f'final loss {float(value):.4f}   train accuracy {acc:.3f}')

final loss 0.0050   train accuracy 1.000


## Porting tensor ops from the einops PyTorch examples

A few favourites, each as a one-line `|>` stage — and each fully differentiable:

- **Channel shuffle** (ShuffleNet, ex 5): `b (c1 c2) h w -> b (c2 c1) h w`
- **Space-to-depth** (GLOW squeeze, ex 22): `b c (h h2) (w w2) -> b (c h2 w2) h w`

In [5]:
img = rng.standard_normal((2, 6, 4, 4))   # 6 channels = c1(2) * c2(3); 4x4 spatial

channel_shuffle = $ |> einops.rearrange($, 'b (c1 c2) h w -> b (c2 c1) h w', c1=2)
space_to_depth  = $ |> einops.rearrange($, 'b c (h h2) (w w2) -> b (c h2 w2) h w', h2=2, w2=2)

print('channel shuffle:', np.asarray(channel_shuffle(img)).shape)
print('space-to-depth :', np.asarray(space_to_depth(img)).shape)

# compose both into a scalar and differentiate end-to-end
loss = $ |> channel_shuffle |> space_to_depth |> np.sum($ ** 2)
g = pg.grad(loss)(img)
g = g[0] if isinstance(g, tuple) else g
print('grad shape     :', np.asarray(getattr(g, 'value', g)).shape)

channel shuffle: (2, 6, 4, 4)
space-to-depth : (2, 24, 2, 2)
grad shape     : (2, 6, 4, 4)


## Multi-head attention (rearrange + einsum)

Examples 13–15 build scaled-dot-product attention by splitting heads with `rearrange` and
contracting with `einsum`. The same code differentiates in pycograd — here we take the
gradient of the output energy w.r.t. the input.

In [6]:
seq = rng.standard_normal((1, 3, 8))   # (batch, length, head(2) * dim(4))

def attention_energy(z):
    q = z |> einops.rearrange($, 'b l (head k) -> head b l k', head=2)
    k = z |> einops.rearrange($, 'b t (head k) -> head b t k', head=2)
    v = z |> einops.rearrange($, 'b t (head d) -> head b t d', head=2)
    scores = einops.einsum(q, k, 'head b l k, head b t k -> head b l t')
    attn = scores |> np.exp($) |> $a / np.sum($a, axis=-1, keepdims=True)
    out = einops.einsum(attn, v, 'head b l t, head b t d -> head b l d')
    return out |> einops.rearrange($, 'head b l d -> b l (head d)') |> np.sum($ ** 2)

val, g = pg.value_and_grad(attention_energy)(seq)
grad = g[0] if isinstance(g, tuple) else g
print('attention energy:', float(val))
print('grad norm       :', float(np.linalg.norm(np.asarray(getattr(grad, 'value', grad)))))

attention energy: 28.346849098330914
grad norm       : 14.167105060153746


## Composing with `vmap`

Because einops ops stay on the tape, they also batch: `vmap` maps a per-example einops
pipeline over a stack of inputs.

In [7]:
batch = rng.standard_normal((5, 3, 8, 8))   # 5 examples
per_example = $ |> einops.rearrange($, 'c h w -> (c h w)') |> np.sum($ ** 2)
out = pg.vmap(per_example)(batch)
print('per-example energies:', np.asarray(getattr(out, 'value', out)))

per-example energies: [165.00476872 175.35976966 202.84966466 174.80025766 219.54048986]
